# 消息

### 消息发送

原始查询:
```python
response = model.invoke([
    {
        "role": "system",
        "content": "你是一名渗透测试助理. 请以 penetration test 角色回答问题."
    },
    {
        "role": "user",
        "content": "请使用Markdown格式输出结果, 并且在结果中添加一个表格, 表格中包含问题, 描述, 测试步骤, 测试结果, 漏洞等级, 漏洞类型, 漏洞影响, 漏洞来源, 漏洞修复建议, 漏洞参考链接}
    },
    {
        "role": "assistant",
        "content": "请使用Markdown格式输出结果, 并且在结果中添加一个表格, 表格中包含问题, 描述, 测试步骤, 测试结果, 漏洞等级, 漏洞类型, 漏洞影响, 漏洞来源, 漏洞修复建议, 漏洞参考链接"
    },
    {
        "role": "user",
        "content": "请使用Markdown格式输出结果, 并且在结果中添加一个表格, 表格中包含问题, 描述,}
    }
]
)
```  

BaseMessage类提供了一些常用的方法，SystemMessage类继承自BaseMessage类，用于表示系统消息。HumanMessage类继承自BaseMessage类，用于表示用户消息。AIMessage类继承自BaseMessage类，用于表示AI消息。ToolMessage类继承自BaseMessage类，用于表示工具消息。  

新查询如下:

In [4]:
from langchain.agents import create_agent
from langchain.tools import tool
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

# 获取环境变量中的API密钥和基础URL
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash", #模型名称
    #model ="qwen3.5-flash", #模型切换为qwen3.5
    model_provider="openai", #因为不支持百炼,所以使用兼容的openai接口
    temperature=0.1,
    api_key=api_key,
    base_url=base_url
)

@tool
# 查询nmap命令使用帮助工具
def get_nmap_help(command: str) -> str:
    """获取nmap命令的帮助信息"""
    return f"nmap中{command}的命令作用是查看扫描结果"

agent = create_agent(
    model=model,
    tools=[get_nmap_help]
)

response = agent.invoke(
    { "messages": [
    SystemMessage("你是一名渗透测试助理"),
    HumanMessage("nmap中-v 的命令作用是什么？"),
    AIMessage("nmap中-v 的命令作用是查看扫描结果"),
    HumanMessage("请将结果翻译成中文")
    ]
    }
)
print(response)

for message in response['messages']:
    message.pretty_print()

{'messages': [SystemMessage(content='你是一名渗透测试助理', additional_kwargs={}, response_metadata={}, id='03abc8e2-b305-4ee6-9384-1863d65ae560'), HumanMessage(content='nmap中-v 的命令作用是什么？', additional_kwargs={}, response_metadata={}, id='82ed77ee-fd8b-41eb-bfc5-0f36358179a7'), AIMessage(content='nmap中-v 的命令作用是查看扫描结果', additional_kwargs={}, response_metadata={}, id='eebeec8e-473e-4174-88a0-38d86d6f1100', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='请将结果翻译成中文', additional_kwargs={}, response_metadata={}, id='abcb8a99-a3b2-4b95-9238-0a8b6087bac9'), AIMessage(content='让我先获取nmap中关于 `-v` 参数的详细帮助信息。\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 311, 'total_tokens': 431, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 55, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 

### 多模态模型调用

In [5]:
# 初始化模型
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="qwen3.5-flash",
    model_provider="openai", #因为不支持百炼,所以使用兼容的openai接口
    temperature=0.1,
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

In [7]:
# 创建一个agent,使用qwen3.5模型
agent = create_agent(model= model)

In [12]:
# 创建一条消息,消息内容是一个url链接,该链接是一张图片,消息格式使用HumanMessage

message = HumanMessage(
    [
        {
            "type": "text",
            "text": "描述一下图片内容"
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "https://www.baidu.com/img/flexible/logo/pc/result.png"
            }
        }
    ]
)

stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    print(chunk.content,end="", flush=True)

这张图片展示的是中国知名搜索引擎公司**百度（Baidu）**的官方标志（Logo）。

以下是图片的具体细节描述：

1.  **整体布局**：Logo 呈水平排列，主要由左侧的英文、中间的图形以及右侧的中文组成。
2.  **左侧部分**：是红色的英文字母 **"Bai"**，字体粗壮且为无衬线体。
3.  **中间部分**：是一个深蓝色的**熊掌印**图案。在这个熊掌印的下方（掌心位置），巧妙地嵌入了白色的英文字母 **"du"**。这一设计将“熊掌”的形象与“度”字的发音结合在了一起。
4.  **右侧部分**：是红色的中文字符 **"百度"**，字体风格与左侧的 "Bai" 保持一致。
5.  **配色**：整个标志采用了经典的**红蓝配色**，背景为纯白色。

总的来说，这是一个将英文拼音首字母、中文名称以及具有辨识度的动物形象（熊掌）相结合的品牌标识。